# Functional neighbourhoods in NYC's Citi Bike network

**DTU 02807 · Autumn 2026**  
**Authors and student numbers:** Add group members before hand-in.

Citi Bike stations can be grouped by the trips connecting them or by the hourly patterns of bikes arriving and leaving. This notebook studies both views using trip histories from September 2025 through August 2026.

1. Do communities derived from trips retain more trips than a geographic partition with the same number of groups?
2. How much trip flow stays inside the full-year communities at different times of day?
3. Do station net-flow profiles form distinct groups or vary gradually along a commuting pattern?

Graph clustering and profile clustering cover the course methods. **Directed Infomap and the map equation** are the method beyond the curriculum. The analyses below are a first pass for the final hand-in.

## 1. Setup and data

Use the repository's Python environment. The README explains how to supply the prepared Parquet tables or rebuild them from the public trip files. Running this notebook only reads the prepared tables.

The pipeline standardizes station IDs, handles overlapping monthly files, filters non-NYC and operator stations, and checks trip durations and speeds. Each graph edge counts trips between distinct stations. Hourly station profiles count trip starts and ends at their respective local timestamps, including round trips. The cleaning and feature code is in `src/citibike`.

Source: [Citi Bike System Data](https://citibikenyc.com/system-data).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from citibike import graph, profiles
from citibike.paths import PROCESSED

SEED = 42
INFOMAP_TRIALS = 2  # Repeat across seeds and use more trials in the final evaluation.
PROFILE_K_VALUES = range(2, 7)

required = ("stations.parquet", "edges.parquet", "profiles.parquet")
missing = [name for name in required if not (PROCESSED / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing prepared tables under {PROCESSED}: {', '.join(missing)}. "
        "See Prepare the data in README.md."
    )

In [ ]:
stations = pd.read_parquet(PROCESSED / "stations.parquet").set_index("station_id")
cleaning_report = PROCESSED / "cleaning_report.csv"
if cleaning_report.is_file():
    display(pd.read_csv(cleaning_report))

display(stations[["name", "arrivals", "departures", "first_seen", "last_seen"]].head())
print(f"Stations in the prepared table: {len(stations):,}")

## 2. Trip network and community detection

Let \(W_{ij}\) count trips from station \(i\) to \(j\). We keep the largest weakly connected component of the full-year graph after applying the station activity threshold in `graph.node_set`. It uses an average of at least one recorded departure or arrival per day between first and last observation.

Spectral clustering and Louvain use \(A = W + W^\top\). Spectral clustering applies k-means to the row-normalized leading eigenvectors of the normalized adjacency matrix. Louvain seeks a high-modularity partition.

**Infomap** uses a random walk on the directed, weighted graph. The map equation measures how many bits per step are needed to encode the walk with a community codebook and local station codebooks. Communities with frequent internal movement allow the walker to reuse local codes. See [Rosvall and Bergstrom (2008)](https://doi.org/10.1073/pnas.0706851105).

In [ ]:
annual = graph.load_graph("all")
coordinates = stations.loc[annual.ids, ["lng", "lat"]].copy()
# Correct the longitude scale at the dataset's mean latitude.
longitude_scale = np.cos(np.deg2rad(coordinates["lat"].mean()))
geography = coordinates.assign(lng=coordinates["lng"] * longitude_scale)

print(f"{len(annual.ids):,} stations in the graph")
print(f"{annual.W.nnz:,} directed station pairs")
print(f"{int(annual.W.sum()):,} trips on those edges")

In [ ]:
infomap_labels = graph.infomap(annual.W, seed=SEED, trials=INFOMAP_TRIALS)
infomap_k = np.unique(infomap_labels).size
louvain_labels = graph.louvain(annual.A, seed=SEED)
louvain_k = np.unique(louvain_labels).size

partitions = {
    "Directed Infomap": infomap_labels,
    "Spectral (Infomap k)": graph.spectral_clustering(annual.A, infomap_k, seed=SEED),
    "Geographic k-means (Infomap k)": KMeans(
        n_clusters=infomap_k, n_init=20, random_state=SEED
    ).fit_predict(geography),
    "Louvain": louvain_labels,
    "Geographic k-means (Louvain k)": KMeans(
        n_clusters=louvain_k, n_init=20, random_state=SEED
    ).fit_predict(geography),
}

## 3. First comparison and time-of-day evaluation

Retention is the share of trips with both endpoints inside the same community. Larger communities generally retain more trips, so compare each network partition with a geographic k-means partition at the same $k$. Report modularity on the symmetrized graph at resolution 1 for a second view. Both scores below use the full-year graph used to fit the network partitions, so they describe in-sample structure.

For the time slices, keep the annual Infomap labels fixed and score the trips that start during each period. This measures how well the annual boundaries contain different flows. A later comparison of separately fitted slice partitions can test whether boundaries shift.

In [ ]:
community_scores = pd.DataFrame([
    {
        "method": method,
        "communities": np.unique(labels).size,
        "retention": graph.flow_retention(annual.W, labels),
        "modularity": graph.modularity(annual.A, labels),
    }
    for method, labels in partitions.items()
]).set_index("method")

display(community_scores.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    coordinates["lng"], coordinates["lat"], c=infomap_labels,
    cmap="tab20", s=7, linewidths=0,
)
ax.set(
    title="Full-year Infomap communities",
    xlabel="Longitude (degrees)", ylabel="Latitude (degrees)",
    aspect=1 / longitude_scale,
)
plt.show()

In [ ]:
slice_names = ["weekday_am", "weekday_midday", "weekday_pm", "weekday_night", "weekend"]
labels_by_station = pd.Series(infomap_labels, index=annual.ids)
edges = pd.read_parquet(
    PROCESSED / "edges.parquet",
    filters=[("slice", "in", slice_names)],
)
src_labels = edges["src"].map(labels_by_station)
dst_labels = edges["dst"].map(labels_by_station)
valid = src_labels.notna() & dst_labels.notna()

slice_scores = (
    edges.loc[valid, ["slice", "trips"]]
    .assign(internal_trips=lambda df: df["trips"].where(src_labels.eq(dst_labels), 0))
    .groupby("slice")[["trips", "internal_trips"]]
    .sum()
    .reindex(slice_names)
)
slice_scores["retention"] = slice_scores["internal_trips"] / slice_scores["trips"]
display(slice_scores.round(3))

## 4. Station net-flow profiles

Net flow is arrivals minus departures. For each station, use 24 mean hourly weekday values and 24 mean hourly weekend values, normalized by the station's mean daily arrival-plus-departure activity for that day type. Federal holidays count with weekends in the current aggregation.

Compare **k-means** and **Ward hierarchical clustering** on all 48 features. A higher silhouette and a lower Davies–Bouldin score indicate clearer groups. Inspect cluster sizes alongside these scores. PCA visualizes the same profiles in two dimensions.

In [ ]:
X = profiles.profile_matrix(annual.ids)
profile_scores = profiles.k_selection(X, ks=PROFILE_K_VALUES)
display(profile_scores.round(3))

# Use the highest-silhouette k-means partition for this exploratory plot.
best_kmeans = (
    profile_scores.loc[profile_scores["method"].eq("kmeans")]
    .sort_values(["silhouette", "k"], ascending=[False, True])
    .iloc[0]
)
profile_labels = profiles.cluster(X, int(best_kmeans["k"]), seed=SEED)

pca = PCA(n_components=2)
embedding = pca.fit_transform(X)
pc1, pc2 = pca.explained_variance_ratio_
fig, ax = plt.subplots(figsize=(7, 4))
for label in np.unique(profile_labels):
    mask = profile_labels == label
    ax.scatter(embedding[mask, 0], embedding[mask, 1], s=9, alpha=0.55,
               label=f"Group {label + 1}")
ax.set(
    title="Station profiles projected onto PC1 and PC2",
    xlabel=f"PC1 ({pc1:.1%} of variance)",
    ylabel=f"PC2 ({pc2:.1%} of variance)",
)
ax.legend()
plt.show()

In [ ]:
mean_profiles = X.assign(group=profile_labels + 1).groupby("group").mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True, layout="constrained")
for ax, day, prefix in zip(axes, ("Weekday", "Weekend"), ("wd", "we")):
    columns = [f"{prefix}{hour:02d}" for hour in range(24)]
    for label, row in mean_profiles.iterrows():
        ax.plot(range(24), row[columns], label=f"Group {label}")
    ax.axhline(0, color="0.6", linewidth=0.8)
    ax.set(title=day, xlabel="Local hour", xticks=range(0, 24, 4))
axes[0].set_ylabel("Hourly net flow / daily activity")
axes[1].legend()
plt.show()

## 5. Discussion and hand-in checklist

- **Graph communities.** Compare the measured retention and modularity against the matched geographic baselines. Identify areas or stations worth inspecting.
- **Time of day.** Compare weekday morning with evening and weekends. Fit partitions for each slice on a common station set and use adjusted Rand index (ARI) to compare them. Use variation across seeds within a slice as a reference before interpreting differences across slices.
- **Station roles.** Compare silhouette, Davies–Bouldin, cluster sizes, and the mean profiles. Explain whether the PCA projection suggests distinct groups or a continuum.
- **Limits.** Trip-derived net flow measures recorded arrivals and departures. Actual bike and dock availability also depends on starting inventory, capacities, and operator relocations.
- **Hand-in.** Write the group's conclusions from the completed runs and fill in author details and individual contributions. Include the support package and data reconstruction instructions with the notebook.